In [2]:
!nvidia-smi
!tao --help | head -3
!python3 -c "import PIL, numpy; print('ok')"
!ls /usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf
!cat ~/.tao_mounts.json

Fri Nov 20 15:35:54 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 545.23.08              Driver Version: 545.23.08    CUDA Version: 12.3     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:1E.0 Off |                    0 |
| N/A   26C    P0              24W /  70W |    169MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [3]:
!mv "gen_tickets (1).py" gen_tickets.py
!mv "make_spec (1).py" make_spec.py

In [4]:
!rm -rf dataset
!python3 gen_tickets.py --out dataset --n-tickets 400 --crops 3 --preview

Listo: {'train': 960, 'val': 120, 'test': 120} -> /dli/task/dataset


In [5]:
!python3 gen_tickets.py --out dataset --n-tickets 400 --crops 3
!python3 yolo_to_kitti.py --src dataset --dst kitti

Listo: {'train': 960, 'val': 120, 'test': 120} -> /dli/task/dataset
train 960 imágenes
val 120 imágenes
test 120 imágenes


In [6]:
import getpass, subprocess
key = getpass.getpass("NGC API key: ")
print(subprocess.run(["docker","login","nvcr.io","-u","$oauthtoken","--password-stdin"],
      input=key.encode(), capture_output=True).stdout.decode())

NGC API key:  ········


Login Succeeded



In [7]:
import json, os
json.dump({"Mounts":[{"source":os.getcwd(),"destination":"/workspace/tao-experiments"}]},
          open(os.path.expanduser("~/.tao_mounts.json"),"w"), indent=2)

In [8]:
!tao model yolo_v4 --help 2>&1 | tail -15

    GPU's to be
    used.
  --use_amp
    Flag to
    enable Auto
    Mixed
    Precision.
  --log_file LOG_FILE
    Path to the
    output log
    file.

tasks:
  {train,prune,kmeans,inference,export,evaluate,dataset_convert}
2026-11-20 15:49:10,261 [TAO Toolkit] [INFO] nvidia_tao_cli.components.docker_handler.docker_handler 363: Stopping container.


In [9]:
!rm -rf dataset kitti
!python3 gen_tickets.py --out dataset --n-tickets 400 --crops 3
!python3 yolo_to_kitti.py --src dataset --dst kitti

Listo: {'train': 960, 'val': 120, 'test': 120} -> /dli/task/dataset
train 960 imágenes
val 120 imágenes
test 120 imágenes


In [11]:
import getpass, os, subprocess
os.environ["NGC_CLI_API_KEY"] = getpass.getpass("NGC API key: ")
os.environ["NGC_CLI_ORG"] = "nvidia"   # si falla, prueba con la org que muestre ngc.nvidia.com
ngc = os.path.abspath("ngc_assets/ngc-cli/ngc")
r = subprocess.run([ngc,"registry","model","download-version",
                    "nvidia/tao/pretrained_object_detection:resnet18","--dest","pretrained"],
                   capture_output=True, text=True)
print(r.stdout[-800:], r.stderr[-800:])
!find pretrained -name "*.hdf5"

NGC API key:  ········


…                   …                                                    
  ━━ • … • Remaining: 0… • … • Elapsed: 0… • Total: 1 - Completed: 1 - Failed: 0
       …                   …                                                    

--------------------------------------------------------------------------------
   Download status: COMPLETED
   Downloaded local path model: /dli/task/pretrained/pretrained_object_detection_vresnet18
   Total files downloaded: 1
   Total transferred: 88.96 MB
   Started at: 2026-11-20 15:51:47
   Completed at: 2026-11-20 15:51:48
   Duration taken: 0s
--------------------------------------------------------------------------------
 
pretrained/pretrained_object_detection_vresnet18/resnet_18.hdf5


In [12]:
!tao model yolo_v4 train -e /workspace/tao-experiments/specs/yolo_v4_train.txt \
  -r /workspace/tao-experiments/results -k tickets2026 --gpus 1 \
  2>&1 | tee train_log.txt

2026-11-20 15:52:16,231 [TAO Toolkit] [INFO] root 160: Registry: ['nvcr.io']
2026-11-20 15:52:16,334 [TAO Toolkit] [INFO] nvidia_tao_cli.components.instance_handler.local_instance 360: Running command in container: nvcr.io/nvidia/tao/tao-toolkit:5.0.0-tf1.15.5
2026-11-20 15:52:16,346 [TAO Toolkit] [WARNING] nvidia_tao_cli.components.docker_handler.docker_handler 288: 
Docker will run the commands as root. If you would like to retain your
local host permissions, please add the "user":"UID:GID" in the
DockerOptions portion of the "/root/.tao_mounts.json" file. You can obtain your
users UID and GID by using the "id -u" and "id -g" commands on the
terminal.
2026-11-20 15:52:16,346 [TAO Toolkit] [INFO] nvidia_tao_cli.components.docker_handler.docker_handler 301: Printing tty value True
Using TensorFlow backend.
2026-11-20 15:52:17.127310: I tensorflow/stream_executor/platform/default/dso_loader.cc:50] Successfully opened dynamic library libcudart.so.12
2026-11-20 15:52:17,178 [TAO Toolkit] 

In [13]:
!python3 make_spec.py --kitti kitti --out specs/yolo_v4_train.txt --epochs 60 --batch 4

anchors small/mid/big: [(38.99, 20.00), (63.61, 20.00), (75.93, 20.00)] [(88.25, 20.00), (88.25, 20.00), (100.57, 20.00)] [(125.19, 20.00), (297.60, 20.00), (420.74, 20.00)]
spec escrito en specs/yolo_v4_train.txt


In [ ]:
!tao model yolo_v4 train -e /workspace/tao-experiments/specs/yolo_v4_train.txt \
  -r /workspace/tao-experiments/results -k tickets2026 --gpus 1 \
  2>&1 | tee train_log.txt

2026-11-20 15:54:12,218 [TAO Toolkit] [INFO] root 160: Registry: ['nvcr.io']
2026-11-20 15:54:12,311 [TAO Toolkit] [INFO] nvidia_tao_cli.components.instance_handler.local_instance 360: Running command in container: nvcr.io/nvidia/tao/tao-toolkit:5.0.0-tf1.15.5
2026-11-20 15:54:12,322 [TAO Toolkit] [WARNING] nvidia_tao_cli.components.docker_handler.docker_handler 288: 
Docker will run the commands as root. If you would like to retain your
local host permissions, please add the "user":"UID:GID" in the
DockerOptions portion of the "/root/.tao_mounts.json" file. You can obtain your
users UID and GID by using the "id -u" and "id -g" commands on the
terminal.
2026-11-20 15:54:12,322 [TAO Toolkit] [INFO] nvidia_tao_cli.components.docker_handler.docker_handler 301: Printing tty value True
Using TensorFlow backend.
2026-11-20 15:54:13.172619: I tensorflow/stream_executor/platform/default/dso_loader.cc:50] Successfully opened dynamic library libcudart.so.12
2026-11-20 15:54:13,224 [TAO Toolkit] 